# 第 8 章 · 记忆（长期：MemoryService）

`MemoryService` 是**长期记忆**的接口，只有两个方向：

- 写：`add_session_to_memory(session)` —— 把一次会话抽取进长期库
- 读：`search_memory(app_name, user_id, query)` —— 语义检索

书里只给了构造代码。下面**真做一次**：跑一个 agent → 把会话存进记忆 → 再检索回来。
所以你能看到"记忆"到底长什么样。

kernel 选 `Python (agentic-adk)`。

In [1]:
import sys
import uuid
from pathlib import Path

# 【适配】定位仓库根；后端（官方 / 中转站）由 .env 的 ADK_BACKEND 决定
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break
from common import adk_model, quiet_adk_sync_warnings, retry_call
quiet_adk_sync_warnings()

try:
    from google.adk.runners import InMemoryRunner
    from google.genai import types
except ModuleNotFoundError:
    raise RuntimeError(
        "本 notebook 需要内核『Python (agentic-adk)』，当前内核里没有 google.adk。\n"
        "VS Code 里点右上角内核名 → 选 Python (agentic-adk) → 重新运行本格。"
    ) from None


def run_once(agent, message):
    """跑一次，返回 (每个 agent 的最终答复, 结束时的 session.state)。"""
    runner = InMemoryRunner(agent)
    sid = str(uuid.uuid4())
    runner.session_service.create_session_sync(
        app_name=runner.app_name, user_id="u", session_id=sid)
    finals = []
    for ev in retry_call(lambda: list(runner.run(
            user_id="u", session_id=sid,
            new_message=types.Content(role="user", parts=[types.Part(text=message)]))), label="run "):
        if ev.is_final_response() and ev.content:
            t = getattr(ev.content, "text", None) or "".join(
                p.text for p in (ev.content.parts or []) if getattr(p, "text", None))
            finals.append((getattr(ev, "author", "?"), (t or "").strip()))
    sess = runner.session_service.get_session_sync(
        app_name=runner.app_name, user_id="u", session_id=sid)
    return finals, sess.state

import asyncio
import threading

from google.adk.agents import LlmAgent
from google.adk.memory import InMemoryMemoryService

# ---- 书里的两种实现 ----
# ① 内存版：进程内有效，重启即失（适合测试）
memory_service = InMemoryMemoryService()

# 【适配】② Vertex AI RAG 版需要 GCP 项目 + 认证 + RAG Corpus，本机不可用 → 只做说明
try:
    from google.adk.memory import VertexAiRagMemoryService
    print("② VertexAiRagMemoryService 可导入，但需要 GCP 的 RAG Corpus，本机不调用。")
except Exception as e:      # noqa: BLE001
    print("② VertexAiRagMemoryService 不可用:", type(e).__name__)


def run_async(coro):
    """【适配】MemoryService 的方法是 async；notebook 内核里可能已有事件循环 → 独立线程跑。"""
    box = {}

    def _target():
        try:
            box["r"] = asyncio.run(coro)
        except BaseException as e:               # noqa: BLE001
            box["e"] = e

    th = threading.Thread(target=_target)
    th.start()
    th.join()
    if "e" in box:
        raise box["e"]
    return box["r"]


# ---- 真做一次：跑一轮对话 → 写入长期记忆 → 检索 ----
USER = "u1"

talk_agent = LlmAgent(
    name="Talker",
    model=adk_model(),
    instruction="用一句话回答。",
)

runner = InMemoryRunner(talk_agent)
APP = runner.app_name                     # 必须用 runner 自己的 app_name，否则读不到 session
sid = str(uuid.uuid4())
runner.session_service.create_session_sync(app_name=APP, user_id=USER, session_id=sid)
for _ in retry_call(lambda: list(runner.run(
        user_id=USER, session_id=sid,
        new_message=types.Content(role="user", parts=[types.Part(text="我下周要去纽约，喜欢靠窗的座位。")]))),
        label="run "):
    pass
session = runner.session_service.get_session_sync(app_name=APP, user_id=USER, session_id=sid)

run_async(memory_service.add_session_to_memory(session))
print("\n已把这次会话写入长期记忆；session 里有", len(session.events), "个事件")

# ★ 检索是长期记忆真正的难点：InMemoryMemoryService 用的是**朴素词匹配**，
#   中文长句切不开就一个词都命不中。同一份记忆，换个查法结果完全不同：
for query in ("座位", "用户喜欢什么座位？"):
    resp = run_async(memory_service.search_memory(app_name=APP, user_id=USER, query=query))
    hits = getattr(resp, "memories", []) or []
    print(f"\nquery={query!r} -> 命中 {len(hits)} 条")
    for m in hits[:1]:
        text = getattr(m, "content", None)
        text = getattr(text, "text", None) or str(text)
        print("  -", (text or "").strip()[:160])

print("\n（同一份记忆，短查询能命中，长句命中不了 —— 记忆的成败在检索，不在存储）")

② VertexAiRagMemoryService 可导入，但需要 GCP 的 RAG Corpus，本机不调用。



已把这次会话写入长期记忆；session 里有 2 个事件

query='座位' -> 命中 2 条
  - parts=[Part(
  text='我下周要去纽约，喜欢靠窗的座位。'
)] role='user'

query='用户喜欢什么座位？' -> 命中 0 条

（同一份记忆，短查询能命中，长句命中不了 —— 记忆的成败在检索，不在存储）
